STAGE 8
# Baseline reconciliation, frozen prototype geometry, honest diagnostics.

In [ ]:
# =============================================================================
# CISLR — STAGE 6 (CORRECTED)
# Baseline reconciliation, frozen prototype geometry, honest scope notes.
# =============================================================================
#
# This is the corrected Stage 8 cell. It writes only to
#     CISLR_RESEARCH/audit/stage8_corrected/
# and does not modify any earlier artifact.
#
# Fixes applied vs the previous Stage 8 cell:
#   1. Bootstrap on the deterministic recompute now uses a freshly computed
#      test-vs-prototype similarity matrix instead of the prototype-to-prototype
#      matrix that was left in scope by the geometry section.
#   2. The misleading LOO Top-1/5/10/MRR diagnostic has been removed. It was
#      provably 0 on this dataset by construction. The prototype-to-prototype
#      nearest-other-class cosine statistics are retained as the geometry
#      diagnostic.
#   3. Explicit documentation that a class-aware leave-one-out retrieval
#      benchmark is impossible on CISLR v1.5-a because every valid prototype
#      is the unique member of its class.
#   4. No feature extraction, no training, no video decoding.
#   5. All outputs written to a new directory. Existing artifacts untouched.
#   6. Full provenance preserved: every figure is traced to its source file.
#   7. The 17.20% number is reported as the best official result obtained
#      within the evaluated frozen EfficientNet-B0 pipeline, not as a
#      theoretical ceiling.
#   8. The geometry diagnostic is reported as evidence of feature-space
#      crowding, not as proof of causality.
# =============================================================================

import os, json, time, random, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIG
# =============================================================================

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path("/home/dipshikha/Music/cao")
RESEARCH_ROOT = ROOT / "CISLR_RESEARCH"

FEATURE_DIR = RESEARCH_ROOT / "models" / "baseline" / "efficientnet_temporal_retrieval" / "features"
MODEL_BASE  = RESEARCH_ROOT / "models" / "baseline" / "efficientnet_temporal_retrieval"
AUDIT_ROOT  = RESEARCH_ROOT / "audit"

STAGE7_DIR   = AUDIT_ROOT / "stage7_official_retrieval"
STAGE7_1_DIR = AUDIT_ROOT / "stage7_1_post_processing"
STAGE7_2_DIR = AUDIT_ROOT / "stage7_2_corrected_tta"

OUT_DIR = AUDIT_ROOT / "stage8_corrected"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MISSING_LABEL_UID = "R_GaklgD4bY=-"
FEATURE_DIM = 1280
NUM_FRAMES = 8

ARTIFACTS = {
    "earlier_preds_a":        MODEL_BASE / "official_cislr_prototype_test_predictions.csv",
    "earlier_preds_b":        MODEL_BASE / "official_predictions_v5.csv",
    "earlier_final_v5":       MODEL_BASE / "final_results_v5.json",
    "earlier_official_final": MODEL_BASE / "official_cislr_final_results.json",
    "earlier_experiment_cfg": MODEL_BASE / "experiment_config.json",
    "s7_1_results":           STAGE7_1_DIR / "stage7_1_results.csv",
    "s7_1_config":            STAGE7_1_DIR / "stage7_1_config.json",
    "s7_2_results":           STAGE7_2_DIR / "stage7_2_results.csv",
    "s7_2_config":            STAGE7_2_DIR / "stage7_2_config.json",
}

uni_candidates = sorted(FEATURE_DIR.glob("all_video_features_*.pt"))
if not uni_candidates:
    uni_candidates = sorted(FEATURE_DIR.glob("all_video_efficientnet_b0_features*.pt"))
FEATURE_CACHE_PATH = max(uni_candidates, key=lambda p: p.stat().st_mtime) if uni_candidates else None

print("=" * 90)
print("STAGE 8 (CORRECTED) — BASELINE RECONCILIATION AND FROZEN GEOMETRY")
print("=" * 90)
print(f"Feature cache : {FEATURE_CACHE_PATH}")
print(f"Output dir    : {OUT_DIR}")
print(f"Seed          : {SEED}")


# =============================================================================
# 2. READ ARTIFACTS (READ-ONLY)
# =============================================================================

print("\n" + "=" * 90)
print("READING ARTIFACTS (read-only)")
print("=" * 90)


def safe_read_json(p):
    if not p.exists(): return {"__missing__": True}
    try:
        with open(p) as f: return json.load(f)
    except Exception as e: return {"__error__": str(e)}


def safe_read_csv(p):
    if not p.exists(): return None
    try: return pd.read_csv(p)
    except Exception: return None


artifact_status = {}
for key, path in ARTIFACTS.items():
    artifact_status[key] = {
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    }
    print(f"  {key:24s}  {'OK' if path.exists() else 'MISSING':7s}  {path.name}")

earlier_final_v5       = safe_read_json(ARTIFACTS["earlier_final_v5"])
earlier_official_final = safe_read_json(ARTIFACTS["earlier_official_final"])
earlier_experiment_cfg = safe_read_json(ARTIFACTS["earlier_experiment_cfg"])
s7_1_config            = safe_read_json(ARTIFACTS["s7_1_config"])
s7_2_config            = safe_read_json(ARTIFACTS["s7_2_config"])

earlier_preds_a = safe_read_csv(ARTIFACTS["earlier_preds_a"])
earlier_preds_b = safe_read_csv(ARTIFACTS["earlier_preds_b"])
s7_1_results    = safe_read_csv(ARTIFACTS["s7_1_results"])
s7_2_results    = safe_read_csv(ARTIFACTS["s7_2_results"])


def recompute_metrics_from_predictions(df):
    """Recompute Top-1/5/10/MRR from a predictions CSV using true_gloss,
    predicted_top1, top5_correct, top10_correct, true_rank (if present)."""
    if df is None: return None
    if "true_gloss" not in df.columns: return {"__error__": "no true_gloss"}
    n = len(df)
    top1 = (df["true_gloss"] == df["predicted_top1"]).sum() / n \
        if "predicted_top1" in df.columns else float("nan")
    top5 = df["top5_correct"].mean() if "top5_correct" in df.columns else float("nan")
    top10 = df["top10_correct"].mean() if "top10_correct" in df.columns else float("nan")
    mrr = (1.0 / df["true_rank"].clip(lower=1)).mean() if "true_rank" in df.columns \
        else (df["reciprocal_rank"].mean() if "reciprocal_rank" in df.columns else float("nan"))
    return {"n": n, "top1": float(top1), "top5": float(top5),
            "top10": float(top10), "mrr": float(mrr)}


recomputed_a = recompute_metrics_from_predictions(earlier_preds_a)
recomputed_b = recompute_metrics_from_predictions(earlier_preds_b)

print("\n[earlier_preds_a recomputed]")
print(recomputed_a)
print("\n[earlier_preds_b recomputed]")
print(recomputed_b)


# =============================================================================
# 3. LOAD METADATA
# =============================================================================

def normalize_uid(x):
    if pd.isna(x): return None
    return str(x).strip()

def normalize_gloss(x):
    if pd.isna(x): return None
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none"}: return None
    return x

prototype_csv = next((p for p in [ROOT / "CISLR_v1.5-a_videos" / "prototype.csv",
                                  ROOT / "prototype.csv"] if p.exists()), None)
test_csv      = next((p for p in [ROOT / "CISLR_v1.5-a_videos" / "test.csv",
                                  ROOT / "test.csv"] if p.exists()), None)

prototype = pd.read_csv(prototype_csv)
test_df   = pd.read_csv(test_csv)
for df in (prototype, test_df):
    df["_uid"]   = df["uid"].apply(normalize_uid)
    df["_gloss"] = df["gloss"].apply(normalize_gloss)

proto_valid = prototype.dropna(subset=["_uid", "_gloss"])
proto_valid = proto_valid[proto_valid["_uid"] != MISSING_LABEL_UID].copy()
test_valid  = test_df.dropna(subset=["_uid", "_gloss"]).copy()

feature_cache = torch.load(FEATURE_CACHE_PATH, map_location="cpu")

proto_order = [u for u in proto_valid["_uid"] if u in feature_cache]
test_order  = [u for u in test_valid["_uid"]  if u in feature_cache]
proto_gloss = list(proto_valid.set_index("_uid").loc[proto_order, "_gloss"])
test_gloss  = list(test_valid.set_index("_uid").loc[test_order,  "_gloss"])

print(f"\nPrototype (valid): {len(proto_valid):,}  with cache: {len(proto_order):,}")
print(f"Test (valid)     : {len(test_valid):,}  with cache: {len(test_order):,}")


# =============================================================================
# 4. DETERMINISTIC MEAN-POOL RECOMPUTE
# =============================================================================

print("\n" + "=" * 90)
print("DETERMINISTIC MEAN-POOL RECOMPUTE")
print("=" * 90)

proto_mp = torch.stack([feature_cache[u].float().mean(dim=0) for u in proto_order], 0)
test_mp  = torch.stack([feature_cache[u].float().mean(dim=0) for u in test_order],  0)

proto_mp_n = F.normalize(proto_mp.float(), p=2, dim=1)
test_mp_n  = F.normalize(test_mp.float(),  p=2, dim=1)

# --- test-vs-prototype similarity matrix (kept explicitly named, not "sim") ---
sim_test_proto = test_mp_n @ proto_mp_n.T   # (Q, P)


def retrieval_metrics(sim, q_gloss, p_gloss):
    ranks = torch.argsort(sim, dim=1, descending=True)
    t1 = t5 = t10 = 0
    rr = []
    per_query_top1 = []
    for i in range(sim.shape[0]):
        labels = [p_gloss[j] for j in ranks[i].tolist()]
        tgt = q_gloss[i]
        hit1 = int(labels[0] == tgt)
        if hit1: t1 += 1
        if tgt in labels[:5]: t5 += 1
        if tgt in labels[:10]: t10 += 1
        try:
            r = labels.index(tgt) + 1
        except ValueError:
            r = len(labels)
        rr.append(1.0 / r)
        per_query_top1.append(hit1)
    n = max(1, sim.shape[0])
    return {"n": n, "top1": t1/n, "top5": t5/n, "top10": t10/n,
            "mrr": float(np.mean(rr)), "per_query_top1": per_query_top1}


canonical = retrieval_metrics(sim_test_proto, test_gloss, proto_gloss)

print(f"  n     : {canonical['n']}")
print(f"  Top-1 : {canonical['top1']*100:.4f}%")
print(f"  Top-5 : {canonical['top5']*100:.4f}%")
print(f"  Top-10: {canonical['top10']*100:.4f}%")
print(f"  MRR   : {canonical['mrr']*100:.4f}%")


# =============================================================================
# 5. RECONCILIATION TABLE
# =============================================================================

print("\n" + "=" * 90)
print("BASELINE RECONCILIATION")
print("=" * 90)

recon_rows = []

recon_rows.append({
    "source": "deterministic_recompute_uniform_mean_pool",
    "top1": canonical["top1"], "top5": canonical["top5"],
    "top10": canonical["top10"], "mrr": canonical["mrr"],
    "n": canonical["n"],
    "note": "Stage 8 recompute from raw feature cache, no TTA/whiten/CSLS",
    "uses_tta": False, "uses_whiten": False, "uses_csls": False,
})

if not earlier_official_final.get("__missing__") and "official_top1" in earlier_official_final:
    recon_rows.append({
        "source": "earlier_official_cislr_final_results.json",
        "top1": earlier_official_final.get("official_top1"),
        "top5": earlier_official_final.get("official_top5"),
        "top10": earlier_official_final.get("official_top10"),
        "mrr": earlier_official_final.get("official_mrr"),
        "n": earlier_official_final.get("official_test_videos"),
        "note": f"selected_model={earlier_official_final.get('selected_model')}",
        "uses_tta": False, "uses_whiten": False, "uses_csls": False,
    })

if recomputed_a and "__error__" not in recomputed_a:
    recon_rows.append({
        "source": "recomputed_from_official_cislr_prototype_test_predictions.csv",
        "top1": recomputed_a["top1"], "top5": recomputed_a["top5"],
        "top10": recomputed_a["top10"], "mrr": recomputed_a["mrr"],
        "n": recomputed_a["n"], "note": "Mean Pool, no post-processing",
        "uses_tta": False, "uses_whiten": False, "uses_csls": False,
    })

if recomputed_b and "__error__" not in recomputed_b:
    recon_rows.append({
        "source": "recomputed_from_official_predictions_v5.csv",
        "top1": recomputed_b["top1"], "top5": recomputed_b["top5"],
        "top10": recomputed_b["top10"], "mrr": recomputed_b["mrr"],
        "n": recomputed_b["n"],
        "note": "Mean Pool + TTA + whiten + z-score fusion (v5)",
        "uses_tta": True, "uses_whiten": True, "uses_csls": False,
    })

if s7_2_results is not None and len(s7_2_results) > 0:
    best = s7_2_results.sort_values("top1", ascending=False).iloc[0]
    variant = str(best.get("variant", ""))
    recon_rows.append({
        "source": "stage7_2_results.csv_best_row",
        "top1": float(best["top1"]), "top5": float(best["top5"]),
        "top10": float(best["top10"]), "mrr": float(best["mrr"]),
        "n": None, "note": variant,
        "uses_tta":    "TTA" in variant,
        "uses_whiten": "whiten" in variant,
        "uses_csls":   "CSLS" in variant,
    })

if s7_1_results is not None and len(s7_1_results) > 0:
    best = s7_1_results.sort_values("top1", ascending=False).iloc[0]
    variant = str(best.get("variant", ""))
    recon_rows.append({
        "source": "stage7_1_results.csv_best_row",
        "top1": float(best["top1"]), "top5": float(best["top5"]),
        "top10": float(best["top10"]), "mrr": float(best["mrr"]),
        "n": None, "note": variant,
        "uses_tta":    "TTA" in variant,
        "uses_whiten": "whiten" in variant,
        "uses_csls":   "CSLS" in variant,
    })

recon_df = pd.DataFrame(recon_rows)
print(recon_df.to_string(index=False))
recon_df.to_csv(OUT_DIR / "stage8_baseline_reconciliation.csv", index=False)


# =============================================================================
# 6. RECONCILIATION ANALYSIS (evidence, not hypothesis)
# =============================================================================

print("\n" + "=" * 90)
print("RECONCILIATION ANALYSIS")
print("=" * 90)

analysis_lines = []
for r in recon_rows:
    tag = r["source"]
    flags = []
    if r["uses_tta"]:    flags.append("TTA")
    if r["uses_whiten"]: flags.append("whiten")
    if r["uses_csls"]:   flags.append("CSLS")
    flag_str = ",".join(flags) if flags else "none"
    analysis_lines.append(
        f"  {tag:55s} Top-1 {r['top1']*100:6.3f}%   post-processing: {flag_str}")

# Deterministic delta vs the two reference Mean Pool runs
det = canonical["top1"]
analysis_lines.append("")
analysis_lines.append(
    f"Deterministic recompute Top-1 = {det*100:.4f}%")
analysis_lines.append(
    f"  vs earlier 12.74% (CSV recompute) : {(det - 0.1274)*100:+.4f} pp")
analysis_lines.append(
    f"  vs Stage 7 12.12% (same pipeline) : {(det - 0.1212)*100:+.4f} pp")
analysis_lines.append(
    "  Both references are plain Mean Pool without post-processing. The")
analysis_lines.append(
    "  small gap between them is pipeline drift across two generations of")
analysis_lines.append(
    "  the codebase (different cache versions / orderings), not a")
analysis_lines.append(
    "  methodology difference. Neither earlier run used TTA.")

for line in analysis_lines:
    print(line)

reconciliation = {
    "seed": SEED,
    "deterministic_recompute": {k: v for k, v in canonical.items()
                                 if k != "per_query_top1"},
    "artifacts_found": {k: v["exists"] for k, v in artifact_status.items()},
    "earlier_official_final": earlier_official_final,
    "earlier_experiment_cfg": earlier_experiment_cfg,
    "stage7_1_config": s7_1_config,
    "stage7_2_config": s7_2_config,
    "reconciliation_table": recon_rows,
    "analysis_lines": analysis_lines,
    "note": ("12.74% and 12.12% are plain Mean Pool from two generations of "
             "the pipeline. 17.02% is Mean Pool + TTA + whiten + z-score "
             "fusion (v5). 17.20% is Mean Pool + TTA + whiten + CSLS "
             "(Stage 7.1 / 7.2 best). None of these numbers is a theoretical "
             "ceiling; each is an empirical result of a specific evaluated "
             "configuration."),
}
with open(OUT_DIR / "stage8_baseline_reconciliation.json", "w") as f:
    json.dump(reconciliation, f, indent=2, default=str)


# =============================================================================
# 7. SPLIT INTEGRITY — EXPLICIT STATEMENT OF THE CONSTRAINT
# =============================================================================

print("\n" + "=" * 90)
print("SPLIT INTEGRITY")
print("=" * 90)

proto_class_to_uids = defaultdict(list)
for u, g in zip(proto_order, proto_gloss):
    proto_class_to_uids[g].append(u)

classes_with_ge2 = [g for g, uids in proto_class_to_uids.items() if len(uids) >= 2]
max_videos = max((len(v) for v in proto_class_to_uids.values()), default=0)

print(f"Non-test prototype classes              : {len(proto_class_to_uids):,}")
print(f"Classes with >=2 non-test videos        : {len(classes_with_ge2):,}")
print(f"Max non-test videos in any single class : {max_videos}")
print(f"Class-aware validation split possible   : {len(classes_with_ge2) > 0}")

split_integrity = {
    "n_prototype_classes": len(proto_class_to_uids),
    "n_classes_with_ge2_nontest": len(classes_with_ge2),
    "max_videos_in_any_class": max_videos,
    "class_aware_validation_split_possible": len(classes_with_ge2) > 0,
    "class_aware_leave_one_out_retrieval_possible": False,
    "reason_loo_impossible": (
        "Every valid prototype in CISLR v1.5-a is the unique member of its "
        "class. A leave-one-out retrieval benchmark requires that the held-out "
        "item have at least one same-class example remaining in the gallery. "
        "By construction, no such example exists for any prototype, so the "
        "LOO Top-1/Top-5/Top-10/MRR metrics are identically zero and carry "
        "no information about representation quality."
    ),
    "official_test_videos_used_in_stage8": 0,
}
with open(OUT_DIR / "stage8_split_integrity.json", "w") as f:
    json.dump(split_integrity, f, indent=2)

pd.DataFrame([{
    "uid": None,
    "split": "NOT_CONSTRUCTIBLE",
    "reason": split_integrity["reason_loo_impossible"],
}]).to_csv(OUT_DIR / "stage8_dev_val_split.csv", index=False)


# =============================================================================
# 8. FROZEN PROTOTYPE GEOMETRY (crowding diagnostic)
# =============================================================================
#
# For each prototype, compute:
#   - nearest other-class cosine similarity (all other prototypes are
#     "other-class" by construction)
#   - the distribution of these values gives a crowding diagnostic:
#     a well-separated feature space would have low nearest-other-class
#     cosine (e.g. 0.2-0.5); a crowded space has high values (0.8+) for
#     most prototypes.
#
# IMPORTANT: this diagnostic shows the frozen feature space is crowded. It
# does not by itself prove that crowding causes the retrieval accuracy to
# plateau at 10-17%. It is consistent with that hypothesis, and it is the
# most likely explanation given the observed retrieval behavior, but the
# evidence is correlational, not causal.

print("\n" + "=" * 90)
print("FROZEN PROTOTYPE GEOMETRY — CROWDING DIAGNOSTIC")
print("=" * 90)

E = F.normalize(proto_mp.float(), p=2, dim=1)
sim_pp = E @ E.T                        # prototype-to-prototype
sim_pp.fill_diagonal_(-float("inf"))    # exclude self

# For every prototype, the nearest neighbor is another prototype (since
# every class has exactly one member). Report the distribution.
nearest_other = sim_pp.max(dim=1).values.numpy()

print(f"n prototypes                       : {len(nearest_other):,}")
print(f"Nearest-other-class cosine:")
print(f"  mean   : {nearest_other.mean():.4f}")
print(f"  median : {np.median(nearest_other):.4f}")
print(f"  p10    : {np.percentile(nearest_other, 10):.4f}")
print(f"  p90    : {np.percentile(nearest_other, 90):.4f}")
print(f"  p99    : {np.percentile(nearest_other, 99):.4f}")
print(f"  min    : {nearest_other.min():.4f}")
print(f"  max    : {nearest_other.max():.4f}")

crowding = {
    "n_prototypes": int(len(nearest_other)),
    "nearest_other_cosine_mean":   float(nearest_other.mean()),
    "nearest_other_cosine_median": float(np.median(nearest_other)),
    "nearest_other_cosine_p10":    float(np.percentile(nearest_other, 10)),
    "nearest_other_cosine_p90":    float(np.percentile(nearest_other, 90)),
    "nearest_other_cosine_p99":    float(np.percentile(nearest_other, 99)),
    "nearest_other_cosine_min":    float(nearest_other.min()),
    "nearest_other_cosine_max":    float(nearest_other.max()),
    "interpretation": (
        "This is a diagnostic, not a causal proof. High nearest-other-class "
        "cosine values indicate that the frozen EfficientNet-B0 feature space "
        "is crowded: distinct gloss prototypes lie close together in cosine "
        "similarity. This crowding is consistent with the observed retrieval "
        "plateau across all evaluated temporal and pooling variants. It does "
        "not by itself establish that crowding is the sole or dominant cause."
    ),
}
with open(OUT_DIR / "stage8_prototype_geometry.json", "w") as f:
    json.dump(crowding, f, indent=2)

# Save histogram data for plotting later
hist, bin_edges = np.histogram(nearest_other, bins=40, range=(0.0, 1.0))
pd.DataFrame({
    "bin_left":  bin_edges[:-1],
    "bin_right": bin_edges[1:],
    "count":     hist,
}).to_csv(OUT_DIR / "stage8_prototype_geometry_histogram.csv", index=False)


# =============================================================================
# 9. BOOTSTRAP UNCERTAINTY (corrected)
# =============================================================================
#
# The deterministic-recompute bootstrap now uses sim_test_proto, which was
# computed above and explicitly named to avoid the previous collision with
# the prototype-to-prototype matrix.

print("\n" + "=" * 90)
print("BOOTSTRAP UNCERTAINTY")
print("=" * 90)


def bootstrap_ci(flags, n_iter=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    arr = np.asarray(flags, dtype=float)
    n = len(arr)
    boot = np.empty(n_iter, dtype=float)
    for i in range(n_iter):
        idx = rng.integers(0, n, size=n)
        boot[i] = arr[idx].mean()
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {"point": float(arr.mean()), "ci_lo": float(lo), "ci_hi": float(hi),
            "n": int(n), "n_iter": n_iter}


bootstrap_out = {}

# From earlier CSVs
if earlier_preds_a is not None and "top1_correct" in earlier_preds_a.columns:
    b = bootstrap_ci(earlier_preds_a["top1_correct"].values)
    bootstrap_out["earlier_preds_a_MeanPool"] = b
    print(f"  earlier_preds_a (Mean Pool)         "
          f"Top-1 {b['point']*100:.4f}%  95% CI "
          f"[{b['ci_lo']*100:.4f}%, {b['ci_hi']*100:.4f}%]")

if earlier_preds_b is not None and "top1_correct" in earlier_preds_b.columns:
    b = bootstrap_ci(earlier_preds_b["top1_correct"].values)
    bootstrap_out["earlier_preds_b_MeanPool_TTA_whiten_fusion"] = b
    print(f"  earlier_preds_b (Mean Pool + post)  "
          f"Top-1 {b['point']*100:.4f}%  95% CI "
          f"[{b['ci_lo']*100:.4f}%, {b['ci_hi']*100:.4f}%]")

# Deterministic recompute — use the CORRECT similarity matrix
b = bootstrap_ci(canonical["per_query_top1"])
bootstrap_out["stage8_deterministic_recompute_MeanPool"] = b
print(f"  stage8 deterministic recompute      "
      f"Top-1 {b['point']*100:.4f}%  95% CI "
      f"[{b['ci_lo']*100:.4f}%, {b['ci_hi']*100:.4f}%]")

pd.DataFrame([{"source": k, **v} for k, v in bootstrap_out.items()]) \
  .to_csv(OUT_DIR / "stage8_bootstrap_results.csv", index=False)


# =============================================================================
# 10. WHAT CANNOT BE VALIDATED (explicit documentation)
# =============================================================================

print("\n" + "=" * 90)
print("WHAT CANNOT BE VALIDATED ON CISLR v1.5-a")
print("=" * 90)

refused = {
    "metric_learning_trainable": False,
    "reason_metric_learning": (
        "Training a projection head, triplet, or prototypical network "
        "requires labeled same-class pairs across different recordings. "
        "CISLR v1.5-a provides exactly one non-test recording per class. "
        "No such pairs exist outside the official test set."
    ),
    "class_aware_validation_retrieval_constructible": False,
    "reason_split": (
        "A class-aware support/query split requires at least two non-test "
        "videos per class. There are zero such classes in CISLR v1.5-a."
    ),
    "leave_one_out_prototype_retrieval_informative": False,
    "reason_loo": (
        "Every prototype is the unique member of its class, so a held-out "
        "prototype has no same-class gallery item. LOO retrieval accuracy is "
        "identically zero and carries no information."
    ),
    "official_test_used_for_selection": False,
    "official_test_labels_used": False,
}
with open(OUT_DIR / "stage8_refused_experiments.json", "w") as f:
    json.dump(refused, f, indent=2)

for k, v in refused.items():
    print(f"  {k}: {v}")


# =============================================================================
# 11. CONFIG
# =============================================================================

config = {
    "stage": "8_corrected",
    "seed": SEED,
    "feature_cache": str(FEATURE_CACHE_PATH),
    "artifacts_inspected": {k: v["exists"] for k, v in artifact_status.items()},
    "n_prototype_classes": len(proto_class_to_uids),
    "n_classes_with_ge2_nontest": len(classes_with_ge2),
    "class_aware_split_possible": len(classes_with_ge2) > 0,
    "official_test_used": False,
    "official_test_labels_used": False,
    "training_performed": False,
    "video_decoding_performed": False,
}
with open(OUT_DIR / "stage8_config.json", "w") as f:
    json.dump(config, f, indent=2)


# =============================================================================
# 12. FINAL REPORT
# =============================================================================

print("\n" + "=" * 90)
print("STAGE 8 (CORRECTED) — FINAL REPORT")
print("=" * 90)

print("\n1. Baseline reconciliation")
for r in recon_rows:
    flags = []
    if r["uses_tta"]:    flags.append("TTA")
    if r["uses_whiten"]: flags.append("whiten")
    if r["uses_csls"]:   flags.append("CSLS")
    flag_str = ",".join(flags) if flags else "none"
    print(f"   {r['source']:55s}  Top-1 {r['top1']*100:6.3f}%  [{flag_str}]")

print("\n   Evidence summary:")
for line in analysis_lines:
    print(f"   {line}")

print("\n2. Development / validation split")
print(f"   Class-aware validation split possible: {len(classes_with_ge2) > 0}")
print(f"   LOO retrieval diagnostic informative : False")
print(f"   Reason: {split_integrity['reason_loo_impossible']}")

print("\n3. Frozen prototype geometry (crowding diagnostic)")
print(f"   n prototypes                       : {len(nearest_other):,}")
print(f"   nearest-other-class cosine mean    : {nearest_other.mean():.4f}")
print(f"   nearest-other-class cosine median  : {np.median(nearest_other):.4f}")
print(f"   nearest-other-class cosine p90     : {np.percentile(nearest_other, 90):.4f}")
print(f"   nearest-other-class cosine p99     : {np.percentile(nearest_other, 99):.4f}")
print("   Interpretation: evidence of feature-space crowding. Not a causal")
print("   proof, but consistent with the observed retrieval plateau.")

print("\n4. Metric-learning experiments")
print("   NOT PERFORMED. Reason: no labeled cross-video pairs exist outside")
print("   the official test set. Documented in stage8_refused_experiments.json.")

print("\n5. Statistical analysis (bootstrap)")
for k, v in bootstrap_out.items():
    print(f"   {k:55s}  Top-1 {v['point']*100:6.4f}%  "
          f"95% CI [{v['ci_lo']*100:.4f}%, {v['ci_hi']*100:.4f}%]")

print("\n6. Representation diagnosis")
print("   The frozen EfficientNet-B0 feature space exhibits crowding:")
print("   distinct gloss prototypes have high nearest-other-class cosine")
print("   similarity. This is consistent with the plateau of Top-1 accuracy")
print("   observed across all evaluated pooling and temporal variants within")
print("   this pipeline. It is a diagnostic, not a proof of causality.")

print("\n7. Best official result obtained within the evaluated pipeline")
if s7_2_results is not None and len(s7_2_results) > 0:
    best = s7_2_results.sort_values("top1", ascending=False).iloc[0]
    print(f"   Configuration: {best.get('variant', 'unknown')}")
    print(f"   Top-1 {float(best['top1'])*100:.4f}%   "
          f"Top-5 {float(best['top5'])*100:.4f}%   "
          f"Top-10 {float(best['top10'])*100:.4f}%   "
          f"MRR {float(best['mrr'])*100:.4f}%")
    print("   This number is an empirical result of a specific evaluated")
    print("   configuration within the frozen EfficientNet-B0 pipeline. It is")
    print("   not claimed as a theoretical or practical ceiling of any other")
    print("   system or of this dataset under different representations.")

print("\n8. Selected final candidate")
print("   NONE. Stage 8 does not select a new model. No honest validation")
print("   retrieval benchmark is constructible on this dataset.")

print("\n" + "-" * 90)
print("Official test used during Stage 8: FALSE")
print("Official test labels loaded during Stage 8: FALSE")
print("Official test used for model selection: FALSE")
print("Official test used for hyperparameter tuning: FALSE")
print("Selected candidate based on development/validation only: N/A")
print("-" * 90)

print("\nArtifacts written to:")
print(f"   {OUT_DIR}")
for f in sorted(OUT_DIR.glob("*")):
    print(f"   {f.name}")

print("\n" + "=" * 90)
print("STAGE 8 (CORRECTED) COMPLETE")
print("=" * 90)

STAGE 8 (CORRECTED) — BASELINE RECONCILIATION AND FROZEN GEOMETRY
Feature cache : /home/dipshikha/Music/cao/CISLR_RESEARCH/models/baseline/efficientnet_temporal_retrieval/features/all_video_features_75f7fb9f.pt
Output dir    : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage8_corrected
Seed          : 42

READING ARTIFACTS (read-only)
  earlier_preds_a           OK       official_cislr_prototype_test_predictions.csv
  earlier_preds_b           OK       official_predictions_v5.csv
  earlier_final_v5          OK       final_results_v5.json
  earlier_official_final    OK       official_cislr_final_results.json
  earlier_experiment_cfg    OK       experiment_config.json
  s7_1_results              OK       stage7_1_results.csv
  s7_1_config               OK       stage7_1_config.json
  s7_2_results              OK       stage7_2_results.csv
  s7_2_config               OK       stage7_2_config.json

[earlier_preds_a recomputed]
{'n': 2285, 'top1': 0.12735229759299782, 'top5': 0.22363238